<a href="https://colab.research.google.com/github/kubenko-k/Spatial-project/blob/main/Data_exploration/Data_exploration.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Dependencies and files

## Install dependencies

In [ ]:
!pip install scanpy[leiden] anndata2ri scikit-misc
!pip install scanpy[leiden] anndata2ri scikit-misc scvi-tools squidpy gseapy decoupler sc-toolbox --quiet

# Необходимо, чтобы конкретно прогрузился matplotlib
import os


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 189.0/189.0 kB 6.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 169.1/169.1 kB 11.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╸ 4.4/4.4 MB 61.6 MB/s eta 0:00:01
ERROR: Operation cancelled by user
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.0/62.0 kB 6.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.8/87.8 kB 7.7 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 5.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 560.6/560.6 kB 19.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 161.3/161.3 kB 16.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 597.6/597.6 kB 42.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 118.0/118.0 kB 12.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## Imports

In [ ]:
import warnings
import scanpy as sc
#import squidpy as sq
import anndata as an
import pandas as pd
import numpy as np
import matplotlib as mpl
import matplotlib.pyplot as plt
import seaborn as sns
from urllib import request
import json
import os

sc.settings.set_figure_params(dpi=80)
#sc.set_figure_params(facecolor="white", figsize=(8, 8))
warnings.simplefilter(action='ignore', category=FutureWarning)
sc.settings.verbosity = 3

## Global variables

Нужно подключить свой гугл диск

In [ ]:
pwd

In [ ]:
dir_path = '/content/drive/My Drive/Skoltech/Spatial_project/'

In [ ]:
os.listdir(dir_path)

##  Make pseudobulk

For layer-level processing the spot data was turned into pseudobulks. It was done by summing gene expression counts for all spots of a particular layer in each of the samples and calculating average gene expression.

In [ ]:
from tqdm.notebook import tqdm
expr_dict = dict()
ann_dict = dict()
for file_name in tqdm(os.listdir(dir_path)):
    file = file_name.split('.')[0]
    # read adata object
    adata = sc.read_h5ad(dir_path + file_name)
    adata.obs.label.replace({"L6a": "L6", "L6b": "L6"}, inplace=True)
    # add pseudobulk label
    adata.obs['pb_label'] = adata.obs.label.astype(str) + '.' + adata.obs.sample_id.astype(str)
    # create pseudobulk df
    sample_layer_list = adata.obs.pb_label.unique().tolist()
    pb_list = [adata[adata.obs.pb_label == sample].X.mean(axis=0) for sample in sample_layer_list]
    pb_df = pd.DataFrame(np.concatenate(pb_list).T, columns=sample_layer_list, index=adata.var_names)
    expr_dict[file] = pb_df
    # create annotation file
    columns = ['layer', 'sample_id']
    annotation_list = [sample.split('.') for sample in sample_layer_list]
    annotation = pd.DataFrame(annotation_list, index=sample_layer_list, columns=columns)
    annotation['condition'] = file
    ann_dict[file] = annotation
    #save files
    #pb_df.to_csv(f'drive/MyDrive/Spatial project/results/DE/pseudobulks/expression_{file}.csv')
    #annotation.to_csv(f'drive/MyDrive/Spatial project/results/DE/pseudobulks/annotation_{file}.csv')

## PCA

PCA analysis was performed to evaluate clustering of samples. Based on the results of PCA samples were normalized by logarithm of library size. As the samples came from two different experiments and could result in batch-effect, they were additionally normalized by mean - mean expression across all samples for each of the genes was extracted from each of the genes in each sample. This normalization also would allow us to compare PCA was again performed to check for results of normalization.


In [ ]:
# create adata objects
adata_dict = dict()
for file in expr_dict.keys():
    adata = an.AnnData(expr_dict[file].T)
    adata.obs = ann_dict[file]
    adata_dict[file] = adata

In [ ]:
adata_dict.keys()

In [ ]:
adata = an.concat([adata_dict['human'], adata_dict['spatial_libd_human']], merge='same')
adata

In [ ]:
adata.obs["lib_size"] = adata.X.sum(axis=1)
adata.obs["log_lib_size"] = np.log(adata.obs["lib_size"])

In [ ]:
sc.pp.normalize_total(adata, target_sum=1e4)
sc.pp.log1p(adata)
sc.pp.pca(adata)

In [ ]:
sc.pl.pca(adata, color=['layer', 'condition', 'log_lib_size'], size=200, ncols=3)

Очень сильный батч эффект, конечно

In [ ]:
sc.pl.pca(adata, color=['layer', 'condition'], components = ['1,2','3,4','5,6','7,8'], size=200)

In [ ]:
sc.pl.pca_loadings(adata, components=[1,2,3,4,5,6,7,8])

## Normed pseudobulk

In [ ]:
adata_norm = adata.copy()

In [ ]:
gene_mean_list = []
sample_layer_list = adata.obs.sample_id.unique().tolist()
for sample in sample_layer_list:
    gene_mean_list.append(adata[adata.obs.sample_id == sample].X.mean(axis=0).reshape(-1, 1))

In [ ]:
gene_mean_df = pd.DataFrame(np.concatenate(gene_mean_list, axis=1), columns=sample_layer_list, index=adata.var_names)
gene_mean_df.head()

In [ ]:
adata[adata.obs.sample_id == sample].obs.index.tolist()

In [ ]:
for sample in sample_layer_list:
    columns = adata[adata.obs.sample_id == sample].obs.index.tolist()
    for column in columns:
        adata_norm[column].X = adata_norm[column].X - gene_mean_df.loc[:, sample].values

In [ ]:
np.isnan(adata_norm.X).sum()

## PCA

In [ ]:
#sc.pp.normalize_total(pb_adata, target_sum=1e4)
#sc.pp.log1p(pb_adata)
sc.pp.pca(adata_norm)

In [ ]:
sc.pl.pca(adata_norm, color=['layer', 'condition'], size=200, ncols=2)

In [ ]:
sc.pl.pca_loadings(adata_norm, components=[1,2,3,4,5,6,7,8])

In [ ]:
adata = adata[:, adata.X.sum(axis=0) > 0]
df = pd.DataFrame(adata.X, index=adata.obs_names, columns=adata.var_names)
yo_df_na = df.replace(0,np.nan)
df = yo_df_na.dropna(axis=1,how="any")

In [ ]:

import statsmodels.api as sm
from statsmodels.formula.api import ols
from tqdm.notebook import tqdm
from statsmodels.stats.multitest import multipletests

df['condition'] = adata.obs.condition
df['layer'] = adata.obs.layer
df['condition'] = df['condition'].str.replace('human', 'young')
df['condition'] = df['condition'].str.replace('spatial_libd_young', 'old')
df_foranova = df.copy()
df_foranova.columns = df_foranova.columns.str.replace("-", "_")
df_foranova.columns = df_foranova.columns.str.replace(".", "__")
results = []

for gene in tqdm(df_foranova.columns.tolist()[:-2]):
    formula = f'{gene} ~ condition + layer + condition:layer'
    model = ols(formula, data=df_foranova).fit()
    aov_table = sm.stats.anova_lm(model, typ=2)
    results.append(aov_table.loc['condition:layer'].tolist())
columns = ['sum_sq', 'df', 'F', 'PR(>F)']
res = pd.DataFrame(results, columns=columns, index=yo_df_nz.columns)
mult_test = multipletests(res['PR(>F)'], method='fdr_bh')
res['p_val_adj'] = mult_test[1]
res.to_csv('ANOVA_res_2.csv')